# Neural Turing Machines: Copy & Recall

A Neural Turing Machine (NTM) augments a neural network with an external
memory matrix and differentiable read/write heads. The controller (an LSTM)
learns to use the memory to solve algorithmic tasks.

**Reference:** Graves, Wayne & Danihelka, "Neural Turing Machines" (2014)

**Two tasks:**
- **Copy**: read a sequence of binary vectors, then reproduce it from memory
- **Associative recall**: given key-value pairs, look up a value by its key

**CLI equivalents:** `make example-ntm-copy` and `make example-ntm-associative-recall`
(50000 epochs each for full convergence)

## Architecture

The NTM has four components:
1. **Memory matrix** `N x M` (N rows of M-dimensional vectors)
2. **LSTM controller** (hidden size H)
3. **Read head** (content-based + location-based addressing)
4. **Write head** (interpolation write)

All dimensions are type parameters, checked at compile time.

In [ ]:
:t ntmLayer

In [ ]:
:t NtmState

## Copy Task

The copy task tests whether the NTM can:
1. Write a sequence of binary vectors to memory
2. Read them back in order

**Encoding:** Binary vectors of width W=8, with a delimiter channel (W+1=9 input dims).
The sequence is presented, then a delimiter, then the model must output the sequence.

**Training:** Two-phase — encode (outputs discarded), then decode (loss on targets).
Uses RMSprop with gradient clipping.

We use a tiny config here (N=10, M=5) to demonstrate the API. Full convergence
requires N=128, M=20 and ~50000 epochs.

In [ ]:
:exec do { srand 42;
  ntm <- ntmLayer {inputSize=9, outputSize=8, n=10, m=5, h=20};
  model <- pure (autoName (OutputLayer ntm));
  putStrLn ("Model: " ++ show model);
  opt <- pure (nativeRmsprop 0.0001 0.95 1.0e-8 10.0 0.9);
  (trained, epochs, loss) <- runTraining
    (\m, d => epochTwoPhaseTensor opt d m)
    (copyTaskBinaryBatchVect {w=8} 1 1 5)
    (simpleConfig 500) model;
  putStrLn ("");
  putStrLn ("Trained " ++ show epochs ++ " epochs, final loss: " ++ show loss) }

## Addressing Mechanism

The NTM read/write heads use a multi-step addressing pipeline:

1. **Content addressing**: cosine similarity between key and memory rows
2. **Interpolation gate** `g`: blend content weights with previous weights
3. **Convolutional shift**: circular shift for location-based access
4. **Sharpening** `gamma`: focus the attention distribution

The head parameters are produced by FC layers from the LSTM hidden state:
- `beta` (key strength): softplus
- `g` (interpolation gate): sigmoid
- `gamma` (sharpening): 1 + softplus
- Shift kernel: softmax over a 3-element window

## Associative Recall

The recall task tests content-based addressing. Given K item-pairs
(e.g., 3 pairs of 6-bit vectors), plus a query item from one pair,
the model must output the corresponding paired item.

This requires the NTM to:
1. Store all pairs in memory during encoding
2. Use the query as a content-based lookup key
3. Read the associated value from the correct memory row

## Scaling Up

For full convergence on the copy task:
```bash
make example-ntm-copy --epochs 50000 --lr 0.0001 --clip 10.0
```

Expected results:
- Short sequences (len 1-5): ~98-100% bit accuracy
- Full sequences (len 1-20): ~87-95% bit accuracy

For associative recall:
```bash
make example-ntm-associative-recall --epochs 50000
```

The NTM learns to use content-based addressing for key-value lookup,
generalizing to unseen key combinations.

## PyTorch Comparison

The NTM is a complex architecture. In PyTorch, it typically requires
~300-500 lines for the controller, heads, and addressing logic.
In idris-ml, `ntmLayer` encapsulates all of this, and the type system
ensures the memory dimensions, head parameter widths, and controller
I/O dimensions are consistent.

See `pytorch/torch_ref/scripts/ntm_copy.py` and `ntm_recall.py` for
the full references.

Next: [DNC](dnc.ipynb) — Differentiable Neural Computer, an evolution of the NTM.